# Coastal flood step 32: protected status of mangroves associated with avoided EADs

This notebook overlays mangrove patches with forest reserves and protected areas for the current weighted area-distance signed mangrove attribution results.

For each minimum/maximum damage scenario, it estimates:
- how much mangrove area associated with positive attributed avoided EAD is inside forest reserves and/or protected areas;
- how much is outside those designations;
- how positive attributed avoided EAD is split between protected and unprotected patch area, using area-share allocation for partial overlaps.

Definitions:
- **Avoided-EAD-associated mangroves** are mangrove patches with `Positive_Avoided_EAD_USD_attributed > 0` in the weighted area-distance attribution outputs.
- **Protected** means inside a forest reserve and/or a protected-area polygon from `protected_landcover`.
- Dollar values are patch-level attribution results allocated by protected/unprotected area share; this is not a separate hydrodynamic attribution model.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import GeometryCollection

pd.options.display.max_columns = 120
pd.options.display.float_format = '{:,.3f}'.format

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
coastal_root = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages'
output_dir = coastal_root / 'mangrove_protected_status_weighted_area_distance'
output_dir.mkdir(parents=True, exist_ok=True)

scenario_paths = {
    'minimum': coastal_root / 'results_coastal_minimum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg',
    'maximum': coastal_root / 'results_coastal_maximum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg',
}

forest_reserves_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/forest_reserves.shp'
protected_areas_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/protected_areas.shp'

for required_path in [forest_reserves_path, protected_areas_path, *scenario_paths.values()]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('Output directory:', output_dir)
for scenario_name, path in scenario_paths.items():
    print(scenario_name, path)
print('Forest reserves:', forest_reserves_path)
print('Protected areas:', protected_areas_path)

## Helper functions

In [ ]:
metric_cols = [
    'Positive_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed_abs',
    'Net_Avoided_EAD_USD_attributed',
    'Absolute_Avoided_EAD_USD_attributed',
    'Gross_Avoided_EAD_USD_attributed',
]

status_order = [
    'Outside forest reserves/protected areas',
    'Forest reserve only',
    'Protected area only',
    'Forest reserve and protected area',
]

status_code_lookup = {
    'Outside forest reserves/protected areas': 0,
    'Forest reserve only': 1,
    'Protected area only': 2,
    'Forest reserve and protected area': 3,
}


def make_valid_gdf(gdf):
    out = gdf[gdf.geometry.notna()].copy()
    out = out[~out.geometry.is_empty].copy()
    out['geometry'] = out.geometry.make_valid()
    out = out[out.geometry.notna()].copy()
    out = out[~out.geometry.is_empty].copy()
    return out


def safe_union(gdf):
    if gdf.empty:
        return GeometryCollection()
    try:
        return gdf.geometry.union_all()
    except Exception:
        return gdf.geometry.unary_union


def format_usd(value):
    if pd.isna(value):
        return 'NA'
    value = float(value)
    sign = '-' if value < 0 else ''
    abs_value = abs(value)
    if abs_value >= 1_000_000:
        return f'{sign}US${abs_value / 1_000_000:,.2f} million'
    if abs_value >= 1_000:
        return f'{sign}US${abs_value / 1_000:,.1f} thousand'
    return f'{sign}US${abs_value:,.0f}'


def format_area(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.1f} ha'


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.1f}%'


def is_pct_column(col):
    return (
        col.startswith('Pct_')
        or col.startswith('Min_Pct_')
        or col.startswith('Max_Pct_')
        or '_Pct_' in col
        or col.endswith('_Pct')
    )


def add_readable_columns(df):
    out = df.copy()
    base_cols = [
        c for c in out.columns
        if not c.endswith('_Readable') and not c.endswith('_Label')
    ]
    for col in [c for c in base_cols if c.endswith('_ha') and not is_pct_column(c)]:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[f'{col}_Readable'] = out[col].apply(format_area)
    for col in [
        c for c in base_cols
        if (c.endswith('_USD') or c.endswith('_attributed') or c.endswith('_abs'))
        and not is_pct_column(c)
    ]:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[f'{col}_Readable'] = out[col].apply(format_usd)
    for col in [c for c in base_cols if is_pct_column(c)]:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[f'{col}_Label'] = out[col].apply(format_pct)
    return out


def compute_status_areas(mangroves, forest_reserve_geom, protected_area_geom):
    both_geom = forest_reserve_geom.intersection(protected_area_geom)
    forest_reserve_only_geom = forest_reserve_geom.difference(protected_area_geom)
    protected_area_only_geom = protected_area_geom.difference(forest_reserve_geom)

    out = mangroves.copy()
    out['Patch_Area_m2'] = out.geometry.area
    out['Forest_Reserve_Only_Area_m2'] = out.geometry.apply(lambda geom: geom.intersection(forest_reserve_only_geom).area if not forest_reserve_only_geom.is_empty else 0.0)
    out['Protected_Area_Only_Area_m2'] = out.geometry.apply(lambda geom: geom.intersection(protected_area_only_geom).area if not protected_area_only_geom.is_empty else 0.0)
    out['Forest_Reserve_and_Protected_Area_m2'] = out.geometry.apply(lambda geom: geom.intersection(both_geom).area if not both_geom.is_empty else 0.0)
    protected_piece_area = (
        out['Forest_Reserve_Only_Area_m2']
        + out['Protected_Area_Only_Area_m2']
        + out['Forest_Reserve_and_Protected_Area_m2']
    )
    out['Outside_Protected_Status_Area_m2'] = (out['Patch_Area_m2'] - protected_piece_area).clip(lower=0.0)
    return out


def allocate_patch_metrics_by_status(mangroves, scenario_name, positive_only=True):
    source = mangroves.copy()
    if positive_only:
        source = source[pd.to_numeric(source['Positive_Avoided_EAD_USD_attributed'], errors='coerce').fillna(0.0) > 0].copy()

    piece_map = {
        'Outside forest reserves/protected areas': 'Outside_Protected_Status_Area_m2',
        'Forest reserve only': 'Forest_Reserve_Only_Area_m2',
        'Protected area only': 'Protected_Area_Only_Area_m2',
        'Forest reserve and protected area': 'Forest_Reserve_and_Protected_Area_m2',
    }

    rows = []
    for _, row in source.iterrows():
        patch_area = float(row['Patch_Area_m2'])
        if patch_area <= 0:
            continue
        for status_label, area_col in piece_map.items():
            area_m2 = float(row.get(area_col, 0.0))
            if area_m2 <= 1e-6:
                continue
            area_share = area_m2 / patch_area
            out_row = {
                'Scenario': scenario_name,
                'Mangrove_ID': row['Mangrove_ID'],
                'Parish': row.get('Parish', pd.NA),
                'TYPE': row.get('TYPE', pd.NA),
                'Protected_Status_Code': status_code_lookup[status_label],
                'Protected_Status': status_label,
                'Protected_Union_Status': 'Outside forest reserves/protected areas' if status_label.startswith('Outside') else 'Inside forest reserves/protected areas',
                'Patch_Area_m2': patch_area,
                'Allocated_Area_m2': area_m2,
                'Allocated_Area_ha': area_m2 / 10_000.0,
                'Area_Share_of_Patch': area_share,
                'Positive_Avoided_EAD_Associated': bool(row['Positive_Avoided_EAD_USD_attributed'] > 0),
            }
            for metric_col in metric_cols:
                if metric_col in source.columns:
                    out_row[metric_col] = float(pd.to_numeric(row[metric_col], errors='coerce')) * area_share
            rows.append(out_row)

    return pd.DataFrame(rows)


def summarise_allocations(allocations, group_cols):
    if allocations.empty:
        return pd.DataFrame(columns=group_cols)
    aggregations = {
        'Allocated_Area_ha': ('Allocated_Area_ha', 'sum'),
        'Mangrove_Count': ('Mangrove_ID', lambda values: values.astype(str).nunique()),
    }
    for metric_col in metric_cols:
        if metric_col in allocations.columns:
            aggregations[metric_col] = (metric_col, 'sum')

    summary = allocations.groupby(group_cols, as_index=False).agg(**aggregations)

    scenario_totals = summary.groupby('Scenario')[['Allocated_Area_ha'] + [c for c in metric_cols if c in summary.columns]].transform('sum')
    summary['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area'] = np.where(
        scenario_totals['Allocated_Area_ha'] != 0,
        100.0 * summary['Allocated_Area_ha'] / scenario_totals['Allocated_Area_ha'],
        np.nan,
    )
    for metric_col in metric_cols:
        if metric_col in summary.columns:
            denom = scenario_totals[metric_col].replace({0: np.nan})
            summary[f'Pct_of_Scenario_{metric_col}'] = 100.0 * summary[metric_col] / denom

    return add_readable_columns(summary)


def build_min_max_ranges(summary, group_cols, value_cols):
    scenario_tables = {}
    for scenario_name, prefix in [('minimum', 'Min'), ('maximum', 'Max')]:
        scenario_df = summary.loc[summary['Scenario'] == scenario_name, group_cols + value_cols].copy().set_index(group_cols)
        scenario_tables[prefix] = scenario_df.rename(columns={col: f'{prefix}_{col}' for col in value_cols})
    all_index = scenario_tables['Min'].index.union(scenario_tables['Max'].index)
    combined = pd.concat([scenario_tables['Min'].reindex(all_index).fillna(0), scenario_tables['Max'].reindex(all_index).fillna(0)], axis=1).reset_index()
    for col in value_cols:
        combined[f'{col}_Range_Min'] = combined[[f'Min_{col}', f'Max_{col}']].min(axis=1)
        combined[f'{col}_Range_Max'] = combined[[f'Min_{col}', f'Max_{col}']].max(axis=1)
    return add_readable_columns(combined)

## Load protected layers

In [ ]:
target_crs = 'EPSG:3448'
forest_reserves = make_valid_gdf(gpd.read_file(forest_reserves_path).to_crs(target_crs))
protected_areas = make_valid_gdf(gpd.read_file(protected_areas_path).to_crs(target_crs))

forest_reserve_geom = safe_union(forest_reserves)
protected_area_geom = safe_union(protected_areas)
protected_union_geom = forest_reserve_geom.union(protected_area_geom)

print(f'Forest reserve polygons: {len(forest_reserves):,}')
print(f'Protected area polygons: {len(protected_areas):,}')
print('Protected-area input layers include proposed protected-area records:', bool('PROPOSED_PROTECTED_AREA' in protected_areas.get('LAYER', pd.Series(dtype=object)).astype(str).unique()))

## Allocate avoided-EAD-associated mangrove patches by protected status

In [ ]:
scenario_mangroves = []
positive_allocations = []
all_allocations = []
scenario_qc_rows = []

for scenario_name, gpkg_path in scenario_paths.items():
    mangroves = make_valid_gdf(gpd.read_file(gpkg_path).to_crs(target_crs))
    required_cols = ['Mangrove_ID', 'Positive_Avoided_EAD_USD_attributed', 'Net_Avoided_EAD_USD_attributed', 'Negative_Avoided_EAD_USD_attributed', 'geometry']
    missing_cols = [col for col in required_cols if col not in mangroves.columns]
    if missing_cols:
        raise KeyError(f'{scenario_name} missing columns: {missing_cols}')

    if 'Negative_Avoided_EAD_USD_attributed_abs' not in mangroves.columns:
        mangroves['Negative_Avoided_EAD_USD_attributed_abs'] = mangroves['Negative_Avoided_EAD_USD_attributed'].abs()
    if 'Gross_Avoided_EAD_USD_attributed' not in mangroves.columns:
        mangroves['Gross_Avoided_EAD_USD_attributed'] = (
            mangroves['Positive_Avoided_EAD_USD_attributed'].fillna(0.0)
            + mangroves['Negative_Avoided_EAD_USD_attributed'].abs().fillna(0.0)
        )

    mangroves = compute_status_areas(mangroves, forest_reserve_geom, protected_area_geom)
    mangroves['Scenario'] = scenario_name
    mangroves['Positive_Avoided_EAD_Associated'] = mangroves['Positive_Avoided_EAD_USD_attributed'].fillna(0.0) > 0
    scenario_mangroves.append(mangroves)

    positive_alloc = allocate_patch_metrics_by_status(mangroves, scenario_name, positive_only=True)
    all_alloc = allocate_patch_metrics_by_status(mangroves, scenario_name, positive_only=False)
    positive_allocations.append(positive_alloc)
    all_allocations.append(all_alloc)

    positive_source = mangroves[mangroves['Positive_Avoided_EAD_Associated']].copy()
    scenario_qc_rows.append({
        'Scenario': scenario_name,
        'All_Mangrove_Count': int(len(mangroves)),
        'Avoided_EAD_Associated_Mangrove_Count': int(len(positive_source)),
        'All_Mangrove_Area_ha': float(mangroves['Patch_Area_m2'].sum() / 10_000.0),
        'Avoided_EAD_Associated_Mangrove_Area_ha': float(positive_source['Patch_Area_m2'].sum() / 10_000.0),
        'Input_Positive_Avoided_EAD_USD': float(positive_source['Positive_Avoided_EAD_USD_attributed'].sum()),
        'Allocated_Positive_Avoided_EAD_USD': float(positive_alloc['Positive_Avoided_EAD_USD_attributed'].sum()),
        'Input_Net_Avoided_EAD_USD': float(positive_source['Net_Avoided_EAD_USD_attributed'].sum()),
        'Allocated_Net_Avoided_EAD_USD': float(positive_alloc['Net_Avoided_EAD_USD_attributed'].sum()),
    })

mangrove_patches = gpd.GeoDataFrame(pd.concat(scenario_mangroves, ignore_index=True), geometry='geometry', crs=target_crs)
avoided_ead_allocations = pd.concat(positive_allocations, ignore_index=True)
all_mangrove_allocations = pd.concat(all_allocations, ignore_index=True)
scenario_qc = pd.DataFrame(scenario_qc_rows)
scenario_qc['Positive_Avoided_EAD_Allocation_Difference_USD'] = scenario_qc['Allocated_Positive_Avoided_EAD_USD'] - scenario_qc['Input_Positive_Avoided_EAD_USD']
scenario_qc['Net_Avoided_EAD_Allocation_Difference_USD'] = scenario_qc['Allocated_Net_Avoided_EAD_USD'] - scenario_qc['Input_Net_Avoided_EAD_USD']

patch_file = output_dir / 'coastal_mangrove_patches_with_protected_status_weighted_area_distance.gpkg'
alloc_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_allocations_by_protected_status.csv'
all_alloc_file = output_dir / 'coastal_all_mangrove_area_allocations_by_protected_status.csv'
qc_file = output_dir / 'coastal_mangrove_protected_status_allocation_qc.csv'

mangrove_patches.to_file(patch_file, driver='GPKG')
avoided_ead_allocations.to_csv(alloc_file, index=False)
all_mangrove_allocations.to_csv(all_alloc_file, index=False)
scenario_qc.to_csv(qc_file, index=False)

print('Saved:', patch_file)
print('Saved:', alloc_file)
print('Saved:', all_alloc_file)
print('Saved:', qc_file)
display(add_readable_columns(scenario_qc))

## Protected-status summaries

In [ ]:
union_summary = summarise_allocations(avoided_ead_allocations, ['Scenario', 'Protected_Union_Status'])
detailed_summary = summarise_allocations(avoided_ead_allocations, ['Scenario', 'Protected_Status_Code', 'Protected_Status', 'Protected_Union_Status'])
all_mangrove_union_summary = summarise_allocations(all_mangrove_allocations, ['Scenario', 'Protected_Union_Status'])

union_summary_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_by_protected_union_status.csv'
detailed_summary_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_by_protected_status.csv'
all_mangrove_union_summary_file = output_dir / 'coastal_all_mangrove_area_by_protected_union_status.csv'

union_summary.to_csv(union_summary_file, index=False)
detailed_summary.to_csv(detailed_summary_file, index=False)
all_mangrove_union_summary.to_csv(all_mangrove_union_summary_file, index=False)

print('Saved:', union_summary_file)
print('Saved:', detailed_summary_file)
print('Saved:', all_mangrove_union_summary_file)

display(union_summary)
display(detailed_summary)
display(all_mangrove_union_summary)

## Minimum-maximum ranges and drafting values

In [ ]:
range_value_cols = [
    'Allocated_Area_ha',
    'Mangrove_Count',
    'Positive_Avoided_EAD_USD_attributed',
    'Net_Avoided_EAD_USD_attributed',
    'Negative_Avoided_EAD_USD_attributed_abs',
    'Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area',
    'Pct_of_Scenario_Positive_Avoided_EAD_USD_attributed',
    'Pct_of_Scenario_Net_Avoided_EAD_USD_attributed',
]

union_ranges = build_min_max_ranges(union_summary, ['Protected_Union_Status'], range_value_cols)
detailed_ranges = build_min_max_ranges(detailed_summary, ['Protected_Status_Code', 'Protected_Status', 'Protected_Union_Status'], range_value_cols)
all_mangrove_union_ranges = build_min_max_ranges(
    all_mangrove_union_summary,
    ['Protected_Union_Status'],
    ['Allocated_Area_ha', 'Mangrove_Count', 'Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area'],
)

union_ranges_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_protected_union_min_max_ranges.csv'
detailed_ranges_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_protected_status_min_max_ranges.csv'
all_mangrove_union_ranges_file = output_dir / 'coastal_all_mangrove_area_protected_union_min_max_ranges.csv'

union_ranges.to_csv(union_ranges_file, index=False)
detailed_ranges.to_csv(detailed_ranges_file, index=False)
all_mangrove_union_ranges.to_csv(all_mangrove_union_ranges_file, index=False)

print('Saved:', union_ranges_file)
print('Saved:', detailed_ranges_file)
print('Saved:', all_mangrove_union_ranges_file)
display(union_ranges)
display(detailed_ranges)

In [ ]:
inside = union_ranges[union_ranges['Protected_Union_Status'] == 'Inside forest reserves/protected areas'].iloc[0]
outside = union_ranges[union_ranges['Protected_Union_Status'] == 'Outside forest reserves/protected areas'].iloc[0]
all_inside = all_mangrove_union_ranges[all_mangrove_union_ranges['Protected_Union_Status'] == 'Inside forest reserves/protected areas'].iloc[0]
all_outside = all_mangrove_union_ranges[all_mangrove_union_ranges['Protected_Union_Status'] == 'Outside forest reserves/protected areas'].iloc[0]

drafting_values = pd.DataFrame([
    {
        'Metric': 'Avoided-EAD-associated mangrove area inside reserves/protected areas',
        'Area_Min_ha': inside['Allocated_Area_ha_Range_Min'],
        'Area_Max_ha': inside['Allocated_Area_ha_Range_Max'],
        'Area_Min_Readable': format_area(inside['Allocated_Area_ha_Range_Min']),
        'Area_Max_Readable': format_area(inside['Allocated_Area_ha_Range_Max']),
        'Share_Min_Pct': inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min'],
        'Share_Max_Pct': inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max'],
        'Share_Min_Label': format_pct(inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min']),
        'Share_Max_Label': format_pct(inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max']),
    },
    {
        'Metric': 'Avoided-EAD-associated mangrove area outside reserves/protected areas',
        'Area_Min_ha': outside['Allocated_Area_ha_Range_Min'],
        'Area_Max_ha': outside['Allocated_Area_ha_Range_Max'],
        'Area_Min_Readable': format_area(outside['Allocated_Area_ha_Range_Min']),
        'Area_Max_Readable': format_area(outside['Allocated_Area_ha_Range_Max']),
        'Share_Min_Pct': outside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min'],
        'Share_Max_Pct': outside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max'],
        'Share_Min_Label': format_pct(outside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min']),
        'Share_Max_Label': format_pct(outside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max']),
    },
    {
        'Metric': 'Positive avoided EAD allocated to protected mangrove area',
        'Area_Min_ha': np.nan,
        'Area_Max_ha': np.nan,
        'Area_Min_Readable': format_usd(inside['Positive_Avoided_EAD_USD_attributed_Range_Min']),
        'Area_Max_Readable': format_usd(inside['Positive_Avoided_EAD_USD_attributed_Range_Max']),
        'Share_Min_Pct': inside['Pct_of_Scenario_Positive_Avoided_EAD_USD_attributed_Range_Min'],
        'Share_Max_Pct': inside['Pct_of_Scenario_Positive_Avoided_EAD_USD_attributed_Range_Max'],
        'Share_Min_Label': format_pct(inside['Pct_of_Scenario_Positive_Avoided_EAD_USD_attributed_Range_Min']),
        'Share_Max_Label': format_pct(inside['Pct_of_Scenario_Positive_Avoided_EAD_USD_attributed_Range_Max']),
    },
    {
        'Metric': 'All mangrove area inside reserves/protected areas',
        'Area_Min_ha': all_inside['Allocated_Area_ha_Range_Min'],
        'Area_Max_ha': all_inside['Allocated_Area_ha_Range_Max'],
        'Area_Min_Readable': format_area(all_inside['Allocated_Area_ha_Range_Min']),
        'Area_Max_Readable': format_area(all_inside['Allocated_Area_ha_Range_Max']),
        'Share_Min_Pct': all_inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min'],
        'Share_Max_Pct': all_inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max'],
        'Share_Min_Label': format_pct(all_inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Min']),
        'Share_Max_Label': format_pct(all_inside['Pct_of_Scenario_Avoided_EAD_Associated_Mangrove_Area_Range_Max']),
    },
])

drafting_file = output_dir / 'coastal_mangrove_protected_status_drafting_values.csv'
drafting_values.to_csv(drafting_file, index=False)
print('Saved:', drafting_file)
display(drafting_values)

## Chart

In [ ]:
plot_df = union_summary.copy()
plot_df['Scenario_Label'] = plot_df['Scenario'].str.capitalize()
plot_df['Protected_Union_Status'] = pd.Categorical(
    plot_df['Protected_Union_Status'],
    categories=['Inside forest reserves/protected areas', 'Outside forest reserves/protected areas'],
    ordered=True,
)
plot_df = plot_df.sort_values(['Scenario', 'Protected_Union_Status'])

fig, ax = plt.subplots(figsize=(9.5, 5.8))
scenario_labels = ['minimum', 'maximum']
x = np.arange(len(scenario_labels))
bottom = np.zeros(len(scenario_labels))
colors = {
    'Inside forest reserves/protected areas': '#31a354',
    'Outside forest reserves/protected areas': '#bdbdbd',
}
for status in ['Inside forest reserves/protected areas', 'Outside forest reserves/protected areas']:
    values = [
        float(plot_df.loc[(plot_df['Scenario'] == scenario) & (plot_df['Protected_Union_Status'] == status), 'Allocated_Area_ha'].sum())
        for scenario in scenario_labels
    ]
    ax.bar(x, values, bottom=bottom, label=status, color=colors[status], edgecolor='#3a3a3a', linewidth=0.6)
    bottom += np.array(values)

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in scenario_labels])
ax.set_ylabel('Avoided-EAD-associated mangrove area (ha)')
ax.set_title('Protected status of mangrove area associated with positive avoided EAD')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=True, loc='upper right')
plt.tight_layout()
chart_file = output_dir / 'coastal_mangrove_avoided_ead_associated_area_by_protected_status.png'
fig.savefig(chart_file, dpi=300, bbox_inches='tight')
print('Saved:', chart_file)
plt.show()

## Method metadata

In [ ]:
metadata = pd.DataFrame([
    {
        'Minimum_Attribution_GPKG': str(scenario_paths['minimum']),
        'Maximum_Attribution_GPKG': str(scenario_paths['maximum']),
        'Forest_Reserve_Path': str(forest_reserves_path),
        'Protected_Area_Path': str(protected_areas_path),
        'Avoided_EAD_Associated_Definition': 'Positive_Avoided_EAD_USD_attributed > 0',
        'Protected_Definition': 'inside forest_reserves.shp and/or protected_areas.shp',
        'Partial_Overlap_Method': 'Mangrove patch area and attributed EAD allocated by protected/unprotected area share',
        'Protected_Areas_Include_Proposed_Protected_Area_Records': bool('PROPOSED_PROTECTED_AREA' in protected_areas.get('LAYER', pd.Series(dtype=object)).astype(str).unique()),
        'Caveat': 'Dollar values are patch-level weighted area-distance attributions allocated by patch area share; this is not a separate hydrodynamic attribution model.',
    }
])
metadata_file = output_dir / 'coastal_mangrove_protected_status_method_metadata.csv'
metadata.to_csv(metadata_file, index=False)
print('Saved:', metadata_file)
display(metadata)